# Azure DevOps — Work Item & Bug Activity Reporter

## Overview
This notebook connects to the Azure DevOps REST API and extracts work item activity and bug lifecycle data for a given organisation and project. The results are saved to Delta tables in a Microsoft Fabric Lakehouse for reporting and trend analysis in Power BI.


##### Import Needed Packages and Libraries

In [1]:
import base64
import json
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone, date
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import SparkSession

StatementMeta(, 49c403a8-f3ce-498e-af17-2565d8b4c3d7, 3, Finished, Available, Finished, False)

##### 
 ##### CONFIGURATION — edit these values before running
##### 

In [ ]:
# ============================================================
#  CONFIGURATION — edit these values before running
# ============================================================
ORG             = 'ENTER DEVOPS ORG NAME HERE]'
PROJECT         = '[ENTER DEVOPS PROJECTNAME HERE]'
PAT             = '[INSERT YOUR PAT HERE - CAN ALSO BE REFERENCED FROM KEY VAULT]'
LOOKBACK_DAYS   = 90     # how far back to pull history — increase for longer trends
PARENT_ID       = [ENTER PARENT ID OF ITEM YOU WANT TO LOOK AT]  # Fabric Roadmap Phase 3
# ============================================================

StatementMeta(, 49c403a8-f3ce-498e-af17-2565d8b4c3d7, 4, Finished, Available, Finished, False)

##### Authentication

In [3]:

HEADERS = {
    'Authorization': 'Basic ' + base64.b64encode(f':{PAT}'.encode()).decode(),
    'Content-Type': 'application/json'
}
del PAT

BASE_URL    = f'https://dev.azure.com/{ORG}/{PROJECT}/_apis'
CUTOFF_DATE = (datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
CUTOFF_UTC  = datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)

print(f'Config loaded — pulling Bug history since {CUTOFF_DATE}')


StatementMeta(, 49c403a8-f3ce-498e-af17-2565d8b4c3d7, 5, Finished, Available, Finished, False)

Config loaded — pulling Bug history since 2026-02-07


##### Step 1: Get all Bug IDs 

In [4]:

# We do NOT filter by state here — we want ALL bugs including resolved/closed
# so we can track the full lifecycle
def get_all_bug_ids():
    parent_clause = f"AND [System.Parent] = {PARENT_ID}" if PARENT_ID else ''
    wiql = {
        'query': f"""
            SELECT [System.Id]
            FROM WorkItems
            WHERE [System.WorkItemType] = 'Bug'
            AND [System.CreatedDate] >= '{CUTOFF_DATE}'
            {parent_clause}
            ORDER BY [System.CreatedDate] ASC
        """
    }
    r = requests.post(
        f'{BASE_URL}/wit/wiql?api-version=7.1',
        json=wiql, headers=HEADERS, timeout=30
    )
    r.raise_for_status()
    ids = [item['id'] for item in r.json().get('workItems', [])]
    parent_label = f'under parent {PARENT_ID}' if PARENT_ID else 'all parents'
    print(f'  Found {len(ids)} bugs created since {CUTOFF_DATE} ({parent_label})')
    return ids

StatementMeta(, 49c403a8-f3ce-498e-af17-2565d8b4c3d7, 6, Finished, Available, Finished, False)

##### Step 2: Batch fetch current details

In [5]:

BUG_FIELDS = [
    'System.Id', 'System.Title', 'System.State',
    'System.AssignedTo', 'System.CreatedDate', 'System.ChangedDate',
    'System.AreaPath', 'System.IterationPath',
    'Microsoft.VSTS.Common.Priority',
    'Microsoft.VSTS.Common.Severity',
]

def get_bugs_batch(ids):
    if not ids:
        return []
    results = []
    for chunk in [ids[i:i+200] for i in range(0, len(ids), 200)]:
        r = requests.post(
            f'{BASE_URL}/wit/workitemsbatch?api-version=7.1',
            json={'ids': chunk, 'fields': BUG_FIELDS},
            headers=HEADERS, timeout=30
        )
        r.raise_for_status()
        results.extend(r.json().get('value', []))
    return results

StatementMeta(, 49c403a8-f3ce-498e-af17-2565d8b4c3d7, 7, Finished, Available, Finished, False)

##### Step 3: Pull full revision history per bug

In [6]:

# This reconstructs every state the bug was ever in and when
def get_bug_revisions(item_id):
    r = requests.get(
        f'{BASE_URL}/wit/workItems/{item_id}/updates?api-version=7.1',
        headers=HEADERS, timeout=30
    )
    r.raise_for_status()

    revisions = []
    for rev in r.json().get('value', []):
        fields      = rev.get('fields') or {}
        changed_raw = fields.get('System.ChangedDate', {}).get('newValue')
        if not changed_raw:
            continue

        changed_dt = datetime.fromisoformat(changed_raw.replace('Z', '+00:00'))

        # Grab state at this revision if it changed (or was set for first time)
        state_delta = fields.get('System.State')
        if state_delta:
            new_state = state_delta.get('newValue')
            old_state = state_delta.get('oldValue')
            if new_state:
                revisions.append({
                    'item_id'    : item_id,
                    'revision'   : rev.get('rev'),
                    'changed_dt' : changed_dt,
                    'changed_date': changed_dt.date(),
                    'old_state'  : old_state,
                    'new_state'  : new_state,
                    'changed_by' : rev.get('revisedBy', {}).get('displayName'),
                })

    return revisions

StatementMeta(, 49c403a8-f3ce-498e-af17-2565d8b4c3d7, 8, Finished, Available, Finished, False)

##### Step 4: Reconstruct end-of-day state for each bug 

In [7]:
def reconstruct_daily_states(item_id, created_date, all_revisions, current_state):
    if not all_revisions:
        all_revisions = []

    # Parse created_date if it comes back as a string from ADO
    if isinstance(created_date, str):
        created_date = datetime.fromisoformat(created_date.replace('Z', '+00:00'))

    # Sort revisions ascending
    sorted_revs = sorted(all_revisions, key=lambda x: x['changed_dt'])

    # Start from the later of bug creation date or the lookback cutoff
    start_date = max(
        created_date.date() if hasattr(created_date, 'date') else created_date,
        CUTOFF_UTC.date()
    )
    today = datetime.now(timezone.utc).date()

    daily_states = []
    current      = 'New'

    rev_idx = 0
    for day_offset in range((today - start_date).days + 1):
        day     = start_date + timedelta(days=day_offset)
        day_end = datetime.combine(day, datetime.max.time()).replace(tzinfo=timezone.utc)

        while rev_idx < len(sorted_revs) and sorted_revs[rev_idx]['changed_dt'] <= day_end:
            current = sorted_revs[rev_idx]['new_state']
            rev_idx += 1

        daily_states.append({
            'item_id'   : item_id,
            'state_date': day.isoformat(),
            'state'     : current,
        })

    return daily_states

StatementMeta(, 49c403a8-f3ce-498e-af17-2565d8b4c3d7, 9, Finished, Available, Finished, False)

##### Step 5: Combined fetch per bug for thread pool

In [8]:

def fetch_bug_history(item_id, created_date, current_state):
    revisions    = get_bug_revisions(item_id)
    daily_states = reconstruct_daily_states(item_id, created_date, revisions, current_state)
    return item_id, revisions, daily_states

StatementMeta(, 49c403a8-f3ce-498e-af17-2565d8b4c3d7, 10, Finished, Available, Finished, False)

##### Step 6: Run

In [9]:

print('\nStep 1: Fetching bug IDs...')
ids = get_all_bug_ids()

if not ids:
    raise SystemExit('No bugs found — try increasing LOOKBACK_DAYS.')

print('\nStep 2: Fetching bug details...')
raw_bugs = get_bugs_batch(ids)

def _display_name(v):
    if isinstance(v, dict):
        return v.get('displayName') or v.get('name') or str(v)
    return v

bug_details = {}
for bug in raw_bugs:
    f   = bug.get('fields', {})
    iid = f.get('System.Id')
    bug_details[iid] = {
        'id'           : iid,
        'title'        : f.get('System.Title'),
        'current_state': f.get('System.State'),
        'assigned_to'  : _display_name(f.get('System.AssignedTo')),
        'created_date' : f.get('System.CreatedDate'),
        'changed_date' : f.get('System.ChangedDate'),
        'area_path'    : f.get('System.AreaPath'),
        'iteration'    : f.get('System.IterationPath'),
        'priority'     : f.get('Microsoft.VSTS.Common.Priority'),
        'severity'     : f.get('Microsoft.VSTS.Common.Severity'),
    }

print(f'\nStep 3: Fetching full revision history for {len(ids)} bugs (parallel)...')
all_revisions_flat  = []
all_daily_states    = []

with ThreadPoolExecutor(max_workers=10) as pool:
    futures = {
        pool.submit(
            fetch_bug_history,
            iid,
            bug_details[iid]['created_date'],
            bug_details[iid]['current_state']
        ): iid for iid in ids if iid in bug_details
    }
    for future in as_completed(futures):
        iid, revisions, daily_states = future.result()
        all_revisions_flat.extend(revisions)
        all_daily_states.extend(daily_states)

StatementMeta(, 49c403a8-f3ce-498e-af17-2565d8b4c3d7, 11, Finished, Available, Finished, False)


Step 1: Fetching bug IDs...
  Found 98 bugs created since 2026-02-07 (under parent 39830)

Step 2: Fetching bug details...

Step 3: Fetching full revision history for 98 bugs (parallel)...


##### Step 7 :  Assemble DataFrames

In [10]:

# Bug master — one row per bug with current details
df_bugs = pd.DataFrame(list(bug_details.values()))

# Daily state snapshots — one row per bug per day
df_daily = pd.DataFrame(all_daily_states)

# Join bug metadata onto daily snapshots
df_daily = df_daily.merge(
    df_bugs[['id', 'title', 'assigned_to', 'priority', 'severity', 'area_path', 'created_date']],
    left_on='item_id', right_on='id', how='left'
).drop(columns=['id'])

# State change log — every state transition recorded
df_state_log = pd.DataFrame(all_revisions_flat)
if not df_state_log.empty:
    df_state_log['changed_date'] = df_state_log['changed_date'].astype(str)
    df_state_log = df_state_log.drop(columns=['changed_dt'])
    df_state_log = df_state_log.sort_values(['item_id', 'changed_date']).reset_index(drop=True)

print(f'\n Data assembled')
print(f'   Bugs tracked             : {len(df_bugs)}')
print(f'   Daily state rows         : {len(df_daily)}')
print(f'   State transition records : {len(df_state_log)}')

StatementMeta(, 49c403a8-f3ce-498e-af17-2565d8b4c3d7, 12, Finished, Available, Finished, False)


✅ Data assembled
   Bugs tracked             : 98
   Daily state rows         : 1068
   State transition records : 265


 ##### Step 8: Quick summary preview 

In [11]:

print('\n=== Current State Breakdown ===')
state_counts = df_bugs['current_state'].value_counts().reset_index()
state_counts.columns = ['state', 'count']
display(state_counts)

print('\n=== Daily Snapshot Sample (last 5 days, all bugs) ===')
cutoff_5 = (datetime.now(timezone.utc) - timedelta(days=5)).strftime('%Y-%m-%d')
last_5 = df_daily[df_daily['state_date'] >= cutoff_5]
display(last_5.sort_values(['state_date', 'item_id']))

StatementMeta(, 49c403a8-f3ce-498e-af17-2565d8b4c3d7, 13, Finished, Available, Finished, False)


=== Current State Breakdown ===


SynapseWidget(Synapse.DataFrame, abbe3940-14ce-4b30-a035-8a1dbd68c65f)


=== Daily Snapshot Sample (last 5 days, all bugs) ===


SynapseWidget(Synapse.DataFrame, a0313dab-5729-4f7e-b8a0-1a0817653025)

 ##### Step 9: Write to Lakehouse

In [12]:

spark      = SparkSession.builder.getOrCreate()
SNAPSHOT_UTC = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')

# Table 1: Bug master
df_bugs_out = df_bugs.copy()
df_bugs_out['snapshot_utc'] = SNAPSHOT_UTC
df_bugs_str = df_bugs_out.astype(str).where(df_bugs_out.notna(), other=None)

(spark.createDataFrame(df_bugs_str)
    .write.format('delta')
    .mode('overwrite')
    .option('mergeSchema', 'true')
    .saveAsTable('ado_bug_master'))

print(f'✅ ado_bug_master          — {len(df_bugs_out)} rows  |  snapshot: {SNAPSHOT_UTC}')


# Table 2: Daily state snapshots — full replace each run since we rebuild all days
df_daily_out = df_daily.copy()
df_daily_out['snapshot_utc'] = SNAPSHOT_UTC
df_daily_str = df_daily_out.astype(str).where(df_daily_out.notna(), other=None)

(spark.createDataFrame(df_daily_str)
    .write.format('delta')
    .mode('overwrite')
    .option('mergeSchema', 'true')
    .saveAsTable('ado_bug_daily_states'))

print(f'ado_bug_daily_states    — {len(df_daily_out)} rows  |  snapshot: {SNAPSHOT_UTC}')


# Table 3: State transition log — append so you keep a full audit trail
if not df_state_log.empty:
    df_log_out = df_state_log.copy()
    df_log_out['snapshot_utc'] = SNAPSHOT_UTC
    df_log_str = df_log_out.astype(str).where(df_log_out.notna(), other=None)

    (spark.createDataFrame(df_log_str)
        .write.format('delta')
        .mode('overwrite')
        .option('mergeSchema', 'true')
        .saveAsTable('ado_bug_state_log'))

    print(f'ado_bug_state_log       — {len(df_log_out)} rows  |  snapshot: {SNAPSHOT_UTC}')

StatementMeta(, 49c403a8-f3ce-498e-af17-2565d8b4c3d7, 14, Finished, Available, Finished, False)

✅ ado_bug_master          — 98 rows  |  snapshot: 2026-05-08 22:13 UTC
✅ ado_bug_daily_states    — 1068 rows  |  snapshot: 2026-05-08 22:13 UTC
✅ ado_bug_state_log       — 265 rows  |  snapshot: 2026-05-08 22:13 UTC
